# FFE 预加重原理详解：为什么只在信号跳变处起作用？

## 问题提出

很多同学在初学 SERDES 时都会有一个疑问：

> **FFE (Feed Forward Equalizer) 为什么能在信号跳变处产生过冲，而在平稳区域保持不变？**

这篇文章将通过**数学推导** + **数值实例** + **可视化**，彻底讲清楚 FFE 的工作原理。

---

## 一、FFE 是什么？

FFE（前馈均衡器）是一个 **3 抽头 FIR 滤波器**，结构如下：

```
输入 x[n] -> [c[-1]] -> + -> 输出 y[n]
            [c[0]]  -> +
            [c[1]]  -> +
```

数学表达式：

$y[n] = c_{-1} \cdot x[n+1] + c_0 \cdot x[n] + c_1 \cdot x[n-1]$

其中：
- $c_{-1}$：**Pre-cursor**（预抽头），通常为负值，如 -0.15
- $c_0$：**Main**（主抽头），保证总能量为 1
- $c_1$：**Post-cursor**（后抽头），通常为负值，如 -0.10

系数约束：$c_{-1} + c_0 + c_1 = 1$

## 二、核心问题：FFE 如何计算？

让我们用具体的数字来演示。假设 FFE 系数为：

| 抽头 | 名称 | 系数值 |
|------|------|--------|
| Pre | c[-1] | -0.15 |
| Main | c[0] | 1.25 |
| Post | c[1] | -0.10 |

验证：-0.15 + 1.25 + (-0.10) = 1.0 OK

In [ ]:
# FFE 系数定义
pre = -0.15
post = -0.10
main = 1.0 - pre - post  # 1.25
fir_taps = [pre, main, post]

print(f"FFE 系数：")
print(f"  Pre (c[-1]) = {pre}")
print(f"  Main(c[0])  = {main}")
print(f"  Post(c[1])  = {post}")
print(f"  系数和 = {pre + main + post}")

## 三、三种典型情况分析

### 情况 1：平稳区域（前后比特相同）

输入序列全是 1：[1, 1, 1, 1, 1]

计算第 3 个位置的输出（n=2）：

y[2] = c[-1] * x[3] + c[0] * x[2] + c[1] * x[1]
y[2] = (-0.15) * 1 + (1.25) * 1 + (-0.10) * 1
y[2] = -0.15 + 1.25 - 0.10 = 1.00

**结论：平稳区域输出等于输入，没有变化！**

In [ ]:
import numpy as np
from scipy.signal import lfilter

# 情况 1：全 1 序列（无跳变）
all_ones = np.array([1, 1, 1, 1, 1, 1, 1])
output_ones = lfilter(fir_taps, [1.0], all_ones)

print("="*60)
print("情况 1：平稳区域（全 1 序列，无跳变）")
print("="*60)
print(f"输入序列：{all_ones}")
print(f"输出序列：{output_ones}")
print()
print("逐点计算过程：")
for i in range(2, len(all_ones)):
    x_next = all_ones[i] if i+1 >= len(all_ones) else all_ones[i+1]
    x_curr = all_ones[i]
    x_prev = all_ones[i-1]
    calc = pre * x_next + main * x_curr + post * x_prev
    print(f"  y[{i}] = ({pre})*{x_next} + ({main})*{x_curr} + ({post})*{x_prev} = {output_ones[i]:.2f}")

### 情况 2：信号跳变处（0 -> 1）

输入序列：[0, 0, 0, 1, 1, 1, 1]

在跳变点（第 4 个位置，n=3）：

y[3] = c[-1] * x[4] + c[0] * x[3] + c[1] * x[2]
y[3] = (-0.15) * 1 + (1.25) * 1 + (-0.10) * 0
y[3] = -0.15 + 1.25 - 0 = 1.10

在跳变前一刻（第 3 个位置，n=2）：

y[2] = c[-1] * x[3] + c[0] * x[2] + c[1] * x[1]
y[2] = (-0.15) * 1 + (1.25) * 0 + (-0.10) * 0
y[2] = -0.15 + 0 - 0 = -0.15

**结论：跳变处产生了过冲（-0.15 和 1.10）！**

In [ ]:
# 情况 2：0->1 跳变
transition = np.array([0, 0, 0, 1, 1, 1, 1])
output_trans = lfilter(fir_taps, [1.0], transition)

print("="*60)
print("情况 2：信号跳变处（0 -> 1）")
print("="*60)
print(f"输入序列：{transition}")
print(f"输出序列：{output_trans}")
print()
print("逐点计算过程：")
for i in range(len(transition)):
    x_next = transition[i] if i+1 >= len(transition) else transition[i+1]
    x_curr = transition[i]
    x_prev = transition[i-1] if i > 0 else 0
    calc = pre * x_next + main * x_curr + post * x_prev
    print(f"  y[{i}] = ({pre})*{x_next} + ({main})*{x_curr} + ({post})*{x_prev} = {output_trans[i]:.2f}")

print()
print("关键观察：")
print(f"  跳变前一刻 y[2] = {output_trans[2]:.2f} (输入是 0，输出变成负值！)")
print(f"  跳变时刻   y[3] = {output_trans[3]:.2f} (输入是 1，输出超过 1！)")

### 情况 3：频繁跳变（010101...）

输入序列：[0, 1, 0, 1, 0, 1, 0, 1]

每个位置都在跳变，所以每个位置都会有过冲/下冲！

In [ ]:
# 情况 3：010101（频繁跳变）
alternating = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 1])
output_alt = lfilter(fir_taps, [1.0], alternating)

print("="*60)
print("情况 3：频繁跳变（010101...）")
print("="*60)
print(f"输入序列：{alternating}")
print(f"输出序列：{output_alt}")
print()
print("逐点计算过程：")
for i in range(len(alternating)):
    x_next = alternating[i] if i+1 >= len(alternating) else alternating[i+1]
    x_curr = alternating[i]
    x_prev = alternating[i-1] if i > 0 else 0
    calc = pre * x_next + main * x_curr + post * x_prev
    print(f"  y[{i}] = ({pre})*{x_next} + ({main})*{x_curr} + ({post})*{x_prev} = {output_alt[i]:.2f}")

## 四、三种情况对比总结

| 场景 | 输入序列 | 输出结果 | 说明 |
|------|---------|---------|------|
| 平稳区域 | [1,1,1,1,1] | [1,1,1,1,1] | 系数和为 1，输出不变 |
| 跳变处 | [0,0,0,1,1,1] | [0,0,0,-0.15,1.1,1,1] | 跳变前后出现过冲 |
| 频繁跳变 | [0,1,0,1,0,1] | [0,-0.15,1.25,-0.25,1.25,-0.25] | 每个跳变都有过冲/下冲 |

### 为什么会这样？

关键公式：

y[n] = c[-1] * x[n+1] + c[0] * x[n] + c[1] * x[n-1]

**当 x[n-1] = x[n] = x[n+1] 时：**
y[n] = (c[-1] + c[0] + c[1]) * x[n] = 1 * x[n] = x[n]

**当 x[n-1] != x[n] 时（跳变）：**
y[n] = c[-1] * x[n+1] + c[0] * x[n] + c[1] * x[n-1] != x[n]

由于前后比特不同，各项不能抵消，就会产生过冲！

## 五、可视化对比

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('ggplot')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# 情况 1：平稳区域
axes[0].stem(all_ones, 'b', label='输入')
axes[0].stem(np.arange(len(output_ones))+0.2, output_ones, 'r', label='输出')
axes[0].set_title('情况 1：平稳区域（全 1 序列）- 输出等于输入', fontsize=12)
axes[0].set_ylabel('幅度')
axes[0].set_ylim(-0.2, 1.2)
axes[0].legend()
axes[0].grid(True)

# 情况 2：跳变处
axes[1].stem(transition, 'b', label='输入')
axes[1].stem(np.arange(len(output_trans))+0.2, output_trans, 'r', label='输出')
axes[1].set_title('情况 2：跳变处（0->1）- 出现过冲', fontsize=12)
axes[1].set_ylabel('幅度')
axes[1].set_ylim(-0.5, 1.5)
axes[1].legend()
axes[1].grid(True)

# 情况 3：频繁跳变
axes[2].stem(alternating, 'b', label='输入')
axes[2].stem(np.arange(len(output_alt))+0.2, output_alt, 'r', label='输出')
axes[2].set_title('情况 3：频繁跳变（010101）- 每个跳变都有过冲/下冲', fontsize=12)
axes[2].set_ylabel('幅度')
axes[2].set_ylim(-0.5, 1.5)
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 六、直观理解：FFE 就像一个差分放大器

可以把 FFE 理解为在检测**相邻比特的差异**：

```
如果 x[n+1] == x[n] == x[n-1]：
    输出 = 输入  (系数和为 1，互相抵消)
    
如果 x[n+1] != x[n] 或 x[n] != x[n-1]：
    输出 = 输入 + 过冲  (系数不能抵消，产生额外值)
```

### 波形图示

```
原始信号：  __|---|___|---|___|---|__
                        
FFE 输出：  __|\__/ \__/ \__|\__/ \__
                        
         平稳       过冲  平稳  过冲
```

可以看到：
    - 平坦区域：FFE 输出与输入相同
    - 跳变边缘：FFE 产生过冲（overshoot）和下冲（undershoot）

## 七、FFE 有什么用？

FFE 产生的过冲不是为了好看，而是为了**补偿信道损耗**！

### 信道的问题

物理信道（PCB 走线、电缆）是一个**低通滤波器**：
- 低频分量（平稳部分）-> 通过良好
- 高频分量（跳变边缘）-> 衰减严重

结果：信号变得圆滑，眼图闭合

### FFE 的补偿

FFE 在发送端**预先增强高频分量**（跳变边缘）：
- 跳变处增加过冲
- 经过信道后，过冲被衰减，刚好回到正常值

结果：接收端信号恢复，眼图张开！

In [ ]:
# 完整演示：FFE + 信道
from scipy.signal import lfilter, butter

# 1. 生成测试信号（包含平稳和跳变）
test_signal = np.array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0])

# 2. 应用 FFE
ffe_output = lfilter(fir_taps, [1.0], test_signal)

# 3. 模拟信道（低通滤波）
cutoff = 0.2
b_ch, a_ch = butter(1, cutoff)
ch_output = lfilter(b_ch, a_ch, ffe_output)

# 4. 对比
fig, ax = plt.subplots(figsize=(14, 5))

ax.stem(test_signal, 'b', label='原始信号')
ax.stem(np.arange(len(ffe_output))+0.3, ffe_output, 'g', label='FFE 输出')
ax.stem(np.arange(len(ch_output))+0.6, ch_output, 'r', label='经过信道后')

ax.set_title('FFE 预加重 + 信道传输 完整流程', fontsize=14)
ax.set_ylabel('幅度')
ax.set_ylim(-0.5, 1.5)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察要点：")
print("1. FFE 输出在跳变处有过冲（绿色虚线）")
print("2. 经过信道后，过冲被衰减，波形更接近原始信号（红色实线）")

## 八、交互实验：调整 FFE 系数观察效果

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

@interact(
    pre_tap=widgets.FloatSlider(min=-0.3, max=0.0, step=0.02, value=-0.15, description='Pre:'),
    post_tap=widgets.FloatSlider(min=-0.3, max=0.0, step=0.02, value=-0.1, description='Post:')
)
def visualize_ffe(pre_tap, post_tap):
    main_tap = 1.0 - pre_tap - post_tap
    taps = [pre_tap, main_tap, post_tap]
    
    # 测试信号：包含多种情况
    test = np.array([1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1])
    output = lfilter(taps, [1.0], test)
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    
    # 波形对比
    axes[0].stem(test, 'b', label='输入')
    axes[0].stem(np.arange(len(output))+0.3, output, 'r', label='输出')
    axes[0].set_title(f'FFE 效果 (Pre={pre_tap}, Main={main_tap:.2f}, Post={post_tap})', fontsize=12)
    axes[0].set_ylabel('幅度')
    axes[0].set_ylim(-0.5, 1.5)
    axes[0].legend()
    axes[0].grid(True)
    
    # 系数图示
    axes[1].bar([-1, 0, 1], taps, color=['red', 'green', 'blue'])
    axes[1].set_xticks([-1, 0, 1])
    axes[1].set_xticklabels(['Pre', 'Main', 'Post'])
    axes[1].set_ylabel('系数值')
    axes[1].set_title('FIR 滤波器系数', fontsize=12)
    axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    axes[1].grid(True)
    
    # 标注系数和
    axes[1].text(0.5, max(taps)+0.05, f'系数和 = {sum(taps):.3f}', ha='center')
    
    plt.tight_layout()
    plt.show()
    
    print("观察：")
    print("  - 当 Pre 更负时，跳变前的下冲更大")
    print("  - 当 Post 更负时，跳变后的过冲更大")
    print("  - 平稳区域（连续 1 或连续 0）输出始终为 1 或 0")

## 九、总结

### FFE 为什么只在跳变处起作用？

| 关键点 | 说明 |
|--------|------|
| 系数和为 1 | c[-1] + c[0] + c[1] = 1，保证平稳区域输出不变 |
| 相邻相同时 | x[n-1] = x[n] = x[n+1]，各项乘以系数后求和 = 输入 |
| 相邻不同时 | 跳变处前后比特不同，系数不能抵消，产生过冲/下冲 |

### 核心公式

y[n] = c[-1] * x[n+1] + c[0] * x[n] + c[1] * x[n-1]

当 x[n-1] = x[n] = x[n+1] = v 时：
y[n] = (c[-1] + c[0] + c[1]) * v = 1 * v = v

### FFE 的作用

1. 在发送端**预先增强高频分量**（跳变边缘）
2. 经过信道衰减后，接收端信号更接近原始值
3. 最终效果：**眼图张开，误码率降低**

---

## 延伸阅读

- [SERDES 系统完整流程详解](./serdes 系统详解.ipynb)
- [低通滤波器对方波的影响](./低通滤波器对方波的影响.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yudonghai789/presonal_Blog/blob/master/高速传输 (serdes)/FFE 原理详解.ipynb)